# Fintech Risk & Fraud Analytics

This notebook analyzes synthetic fintech transaction data to identify fraud patterns, engineer risk-related features, train machine learning models, and produce transaction-level risk scores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

df = pd.read_csv("../data/transactions.csv")

print("Dataset shape:", df.shape)
display(df.head())

print("\nMissing values:")
display(df.isnull().sum())

print("\nFraud distribution:")
display(df["fraud"].value_counts())

print("\nFraud rate:")
print(f"{df['fraud'].mean() * 100:.2f}%")


## Exploratory Fraud Analysis

In [ ]:
fraud_by_country = (
    df.groupby("country")["fraud"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(fraud_by_country)

fraud_by_country.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Fraud Rate by Country")
plt.xlabel("Country")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

fraud_by_type = (
    df.groupby("transaction_type")["fraud"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

display(fraud_by_type)

fraud_by_type.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Fraud Rate by Transaction Type")
plt.xlabel("Transaction Type")
plt.ylabel("Fraud Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

fraud_by_hour = (
    df.groupby("transaction_hour")["fraud"]
    .mean()
    .mul(100)
)

fraud_by_hour.plot(
    kind="line",
    marker="o",
    figsize=(10, 5)
)

plt.title("Fraud Rate by Transaction Hour")
plt.xlabel("Hour")
plt.ylabel("Fraud Rate (%)")
plt.tight_layout()
plt.show()


## Feature Engineering

In [ ]:
df["amount_deviation"] = (
    df["transaction_amount"]
    - df["customer_average_amount"]
)

df["amount_ratio"] = (
    df["transaction_amount"]
    / df["customer_average_amount"]
)

df["high_amount_flag"] = (
    df["transaction_amount"]
    > df["customer_average_amount"] * 2
).astype(int)

df["night_transaction"] = (
    (df["transaction_hour"] < 6)
    | (df["transaction_hour"] >= 23)
).astype(int)

df["failed_transaction_ratio"] = (
    df["failed_transactions"]
    / df["customer_transaction_count"]
)

display(df.head())
print("Feature engineering completed.")


In [ ]:
features = [
    "transaction_amount",
    "transaction_hour",
    "customer_transaction_count",
    "customer_average_amount",
    "international_transaction",
    "failed_transactions",
    "amount_deviation",
    "amount_ratio",
    "high_amount_flag",
    "night_transaction",
    "failed_transaction_ratio",
]

X = df[features]
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## Machine Learning Models

In [ ]:
logistic_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        ),
    ]
)

logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)
logistic_probabilities = logistic_model.predict_proba(X_test)[:, 1]

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

random_forest_model.fit(X_train, y_train)

rf_predictions = random_forest_model.predict(X_test)
rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

def evaluate_model(name, y_true, predictions, probabilities):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions),
        "Recall": recall_score(y_true, predictions),
        "F1": f1_score(y_true, predictions),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
    }

results = pd.DataFrame(
    [
        evaluate_model(
            "Logistic Regression",
            y_test,
            logistic_predictions,
            logistic_probabilities
        ),
        evaluate_model(
            "Random Forest",
            y_test,
            rf_predictions,
            rf_probabilities
        ),
    ]
)

display(results.round(3))


In [ ]:
print("Random Forest classification report")

print(
    classification_report(
        y_test,
        rf_predictions
    )
)

cm = confusion_matrix(
    y_test,
    rf_predictions
)

sns.heatmap(
    cm,
    annot=True,
    fmt="d"
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## Transaction Risk Scoring

In [ ]:
risk_results = X_test.copy()

risk_results["actual_fraud"] = y_test.values
risk_results["fraud_probability"] = rf_probabilities
risk_results["risk_score"] = rf_probabilities * 100

risk_results["risk_level"] = pd.cut(
    risk_results["risk_score"],
    bins=[-1, 30, 70, 100],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

risk_results = risk_results.sort_values(
    "risk_score",
    ascending=False
)

display(risk_results.head(20))

risk_results.to_csv(
    "../data/fraud_risk_scores.csv",
    index=False
)

print("Saved: data/fraud_risk_scores.csv")


## Feature Importance

In [ ]:
feature_importance = pd.DataFrame(
    {
        "feature": features,
        "importance": random_forest_model.feature_importances_,
    }
).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

feature_importance.sort_values(
    "importance"
).plot(
    x="feature",
    y="importance",
    kind="barh",
    figsize=(10, 7)
)

plt.title("Feature Importance for Fraud Detection")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## Business Interpretation

The analysis identifies transaction characteristics associated with higher fraud probability.

High-risk transactions can be prioritized for additional investigation or verification.

The model is intended as a decision-support component rather than an autonomous fraud decision system.

Because the dataset is synthetic, all findings must be validated against real transaction data before operational deployment.